# 1.RAPTOR递归文档树优化策略

知识点讲解：RAPTOR 递归文档树检索优化

1. 什么是 RAPTOR（定位与来源）
   - RAPTOR = Recursive Abstractive Processing for Tree-Organized Retrieval，2024 年论文（arXiv 2401.18059）首次提出
   - 用树状结构组织检索的递归抽象处理技术：自下而上对文本块聚类、归纳，形成分层结构
   - 针对长上下文场景：传统 RAG 依赖短连续文本块检索，长上下文需要更好的最小化分块方法
   - 目标是在「多文档、超长上下文、高准确性、超低成本」之间取得平衡

2. 递归构建流程（六步）
   - ① 对原始文本分块，拆成合适大小
   - ② 对文档块嵌入/向量化，存入向量数据库（此时向量处于高维）
   - ③ 将高维向量降维（如降到 2/3 维），降低运算成本
   - ④ 对降维向量聚类，找出同一类的文档组
   - ⑤ 合并文档组文本，用 LLM 摘要汇总得到新文本，然后重复 ②–⑤
   - ⑥ 直到只剩一个文档且长度合适，结束整个流程
   - 本质：叶子是原文块，逐层向上聚类生成父节点（摘要），形成多层抽象

3. 两种检索策略
   - 树遍历（tree traversal）：从根节点逐层向下，每层按查询与节点嵌入的余弦相似度选最相关 top-k，把其子节点作为下一层候选，直到叶节点，拼接所选节点文本
   - 折叠树（collapsed tree）：把整棵树展平为单层，计算查询与所有节点嵌入的余弦相似度选 top-k
   - 论文更推荐折叠树；且折叠树与「向量数据库基础搜索」一样遍历所有向量（同层），
     只需把全部节点存入向量库并执行相似性搜索即可实现

4. 技术栈（降维 + 聚类）
   - UMAP：全局降维，保留向量的局部与全局结构，作为聚类预处理
   - GMM（高斯混合模型）：概率聚类，支持软分配（一个样本可属多个簇）
   - BIC（贝叶斯信息准则）：自动确定最优聚类数

5. 与多向量/表征索引的关系
   - 同属索引阶段「多向量/多表征」思路的延伸，但 RAPTOR 用树状结构的层层摘要
   - 区别在于：多向量是每个块生成若干派生文本（摘要/假设问题），RAPTOR 进一步把派生文本再聚类、再摘要，形成层次

6. 优势
   - 多粒度：同时提供细节（叶子/原文）与宏观视角（各层摘要），既能答具体问题也能答概念性/宏观问题
   - 自适应层次：自动构建合适的树结构
   - 小嵌入模型也能有不错效果：低成本、高性能
   - 通过摘要过滤冗余，减少噪声

7. 局限与成本
   - 构建成本高：每层都要用 LLM 摘要，适合离线构建
   - 更新/新增数据麻烦：树状结构使增删比其它策略复杂得多，外挂文档增加时复杂度成倍递增
   - 与多向量索引一样，额外 LLM 总结会引入信息失真，数据越多失真概率越大（谨慎使用）

8. 数据更新策略（论文未提供，常规做法）
   - 重新构建：新增大量文档时重建整棵树
   - 增量更新：新增不多时只更新部分树，把新块作新叶子，调整上层摘要
   - 利用现有节点：新文档与某些节点相似时，合并进现有节点并重新生成摘要
   - 层次化更新：按新文档重要性决定在树的哪一层更新

9. LangChain 现状与学习建议
   - LangChain 没有内置 RAPTOR 实现；该策略涵盖降维、聚类、递归检索，目前使用较少
   - 重点是理解其运行流程，无需深入实现细节

10. 索引阶段优化策略全景
   - 代表性策略共四类：分割策略（语义分割、递归字符分割）、多向量/表征检索、RAPTOR、特定 Embeddings（用更高维嵌入模型，无复杂技巧）
   - 实际用得最多的仍是「文档加载/分割」与「特定 Embeddings」两类不引入 LLM 改写的策略

11. 应用场景与最佳实践
   - 适用：长文档检索（书籍、报告、论文）、需理解整体结构的场景、既要细节又要概览的问答
   - 层数通常 2–3 层即可，过多会损失细节；聚类参数按文档数量调整；确保摘要保留关键信息


In [ ]:
from typing import Optional
import dotenv
import numpy as np
import pandas
import pandas as pd
import umap
import weaviate
from langchain_unstructured import UnstructuredLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_weaviate import WeaviateVectorStore
from sklearn.mixture import GaussianMixture
from weaviate.auth import AuthApiKey
dotenv.load_dotenv()
RANDOM_SEED = 224
embd = HuggingFaceEmbeddings(
    model_name="thenlper/gte-small",
    cache_folder="./embeddings/",
    encode_kwargs={"normalize_embeddings": True},
)
model = ChatOpenAI(model="deepseek-v4-pro", temperature=0)
db = WeaviateVectorStore(
    client=weaviate.connect_to_wcs(
        cluster_url="https://mbakeruerziae6psyex7ng.c0.us-west3.gcp.weaviate.cloud",
        auth_credentials=AuthApiKey("ZltPVa9ZSOxUcfafelsggGyyH6tnTYQYJvBx"),
    ),
    index_name="RaptorRAG",
    text_key="text",
    embedding=embd,
)

In [ ]:
def global_cluster_embeddings(
        embeddings: np.ndarray, dim: int, n_neighbors: Optional[int] = None, metric: str = "cosine",
) -> np.ndarray:
    """
    使用 UMAP 对传递嵌入向量进行全局降维
    UMAP（Uniform Manifold Approximation and Projection）是一种降维算法，
    它在保留数据的局部和全局结构方面表现优异，特别适合作为聚类的预处理步骤。
    Args:
        embeddings: 需要降维的嵌入向量数组，shape 为 (n_samples, n_features)
        dim: 降低后的目标维度，通常设置为 10-50
        n_neighbors: 每个向量需要考虑的邻居数量，影响局部结构的保留程度
                     如果没有提供，默认为嵌入数量的开方
        metric: 用于 UMAP 的距离度量方式，默认为余弦相似性
    Returns:
        降维到指定维度的 numpy 嵌入数组，shape 为 (n_samples, dim)
    说明：
        全局聚类用于第一轮聚类，使用较少的邻居数，保留全局结构
    """
    if n_neighbors is None:
        n_neighbors = int((len(embeddings) - 1) ** 0.5)
    return umap.UMAP(n_neighbors=n_neighbors, n_components=dim, metric=metric).fit_transform(embeddings)

In [ ]:
def local_cluster_embeddings(
        embeddings: np.ndarray, dim: int, n_neighbors: int = 10, metric: str = "cosine",
) -> np.ndarray:
    """
    使用 UMAP 对嵌入进行局部降维处理，通常在全局聚类之后进行
    局部降维使用更多的邻居数，更注重保留局部结构，
    适合在已经完成全局聚类后，对每个全局聚类内部进行细粒度聚类。
    Args:
        embeddings: 需要降维的嵌入向量数组
        dim: 降低后的目标维度
        n_neighbors: 每个向量需要考虑的邻居数量，默认为 10
        metric: 用于 UMAP 的距离度量方式，默认为余弦相似性
    Returns:
        降维到指定维度的 numpy 嵌入数组
    说明：
        局部聚类用于第二轮聚类，使用固定的邻居数，关注局部结构
    """
    return umap.UMAP(
        n_neighbors=n_neighbors, n_components=dim, metric=metric,
    ).fit_transform(embeddings)

In [ ]:
def get_optimal_clusters(
        embeddings: np.ndarray, max_clusters: int = 50, random_state: int = RANDOM_SEED,
) -> int:
    """
    使用高斯混合模型结合贝叶斯信息准则（BIC）确定最佳的聚类数目
    BIC 是一种模型选择标准，它在模型拟合度和复杂度之间取得平衡。
    BIC 值越小，表示模型越好。通过比较不同聚类数的 BIC 值，
    可以自动找到最优的聚类数，避免手动调参。
    Args:
        embeddings: 需要聚类的嵌入向量数组
        max_clusters: 最大聚类数，用于限制搜索范围
        random_state: 随机数种子，确保结果可复现
    Returns:
        最优聚类数（使 BIC 最小的聚类数）
    原理：
        - BIC = -2 * log-likelihood + k * log(n)
        - k 是参数数量，n 是样本数量
        - BIC 惩罚复杂模型，避免过拟合
    """
    max_clusters = min(max_clusters, len(embeddings))
    n_clusters = np.arange(1, max_clusters)
    bics = []
    for n in n_clusters:
        gm = GaussianMixture(n_components=n, random_state=random_state)
        gm.fit(embeddings)
        bics.append(gm.bic(embeddings))
    return n_clusters[np.argmin(bics)]

In [ ]:
def gmm_cluster(embeddings: np.ndarray, threshold: float, random_state: int = 0) -> tuple[list, int]:
    """
    使用基于概率阈值的高斯混合模型（GMM）对嵌入进行聚类
    与传统的 K-Means 等硬聚类不同，GMM 是软聚类，每个样本可以属于多个聚类。
    通过设置概率阈值，可以让一个文档同时属于多个聚类，
    这在文档涉及多个主题时特别有用。
    Args:
        embeddings: 需要聚类的嵌入向量数组（已降维）
        threshold: 概率阈值，样本属于某聚类的概率超过此值时才分配到该聚类
                   较低的阈值（如 0.1）允许更多的软分配
        random_state: 用于可重现的随机性种子
    Returns:
        包含聚类标签和聚类数目的元组
        - labels：每个样本的聚类标签列表，每个元素是一个数组（软聚类）
        - n_clusters：确定的聚类数目
    示例：
        如果 threshold=0.1，某文档对聚类 0 的概率为 0.7，对聚类 1 的概率为 0.2，
        那么该文档会被分配到 [0, 1] 两个聚类
    """
    n_clusters = get_optimal_clusters(embeddings)
    gm = GaussianMixture(n_components=n_clusters, random_state=random_state)
    gm.fit(embeddings)
    probs = gm.predict_proba(embeddings)
    labels = [np.where(prob > threshold)[0] for prob in probs]
    return labels, n_clusters

In [ ]:
def perform_clustering(embeddings: np.ndarray, dim: int, threshold: float) -> list[np.ndarray]:
    """
    对嵌入进行聚类，首先全局降维，然后使用高斯混合模型进行聚类，
    最后在每个全局聚类中进行局部聚类
    这是一个两级聚类策略：
    1. 全局聚类：将所有文档分成大类（如"技术文档"、"产品介绍"等）
    2. 局部聚类：在每个大类内部进一步细分（如"API文档"、"配置文档"等）
    Args:
        embeddings: 需要执行操作的嵌入向量列表
        dim: 指定的降维维度，通常设置为 10
        threshold: 概率阈值，用于 GMM 软聚类
    Returns:
        包含每个嵌入的聚类 ID 的列表，每个数组代表一个嵌入的聚类标签
    示例：
        如果某文档在全局聚类中属于聚类 0 和 1，在聚类 0 内部属于局部聚类 0，
        在聚类 1 内部属于局部聚类 1，那么最终标签可能是 [0, 3]
        （假设聚类 0 有 3 个局部聚类）
    """
    if len(embeddings) <= dim + 1:
        return [np.array([0]) for _ in range(len(embeddings))]
    reduced_embeddings_global = global_cluster_embeddings(embeddings, dim)
    global_clusters, n_global_clusters = gmm_cluster(reduced_embeddings_global, threshold)
    all_local_clusters = [np.array([]) for _ in range(len(embeddings))]
    total_clusters = 0
    for i in range(n_global_clusters):
        global_cluster_embeddings_ = embeddings[
            np.array([i in gc for gc in global_clusters])
        ]
        if len(global_cluster_embeddings_) == 0:
            continue
        if len(global_cluster_embeddings_) <= dim + 1:
            local_clusters = [np.array([0]) for _ in global_cluster_embeddings_]
            n_local_clusters = 1
        else:
            reduced_embeddings_local = local_cluster_embeddings(global_cluster_embeddings_, dim)
            local_clusters, n_local_clusters = gmm_cluster(reduced_embeddings_local, threshold)
        for j in range(n_local_clusters):
            local_cluster_embeddings_ = global_cluster_embeddings_[
                np.array([j in lc for lc in local_clusters])
            ]
            indices = np.where(
                (embeddings == local_cluster_embeddings_[:, None]).all(-1)
            )[1]
            for idx in indices:
                all_local_clusters[idx] = np.append(all_local_clusters[idx], j + total_clusters)
        total_clusters += n_local_clusters
    return all_local_clusters

In [ ]:
def embed(texts: list[str]) -> np.ndarray:
    """
    将传递的文本列表转换成嵌入向量列表
    Args:
        texts: 需要转换的文本列表
    Returns:
        生成的嵌入向量列表并转换成 numpy 数组
    说明：
        使用全局定义的 embd 嵌入模型（HuggingFace gte-small）
    """
    text_embeddings = embd.embed_documents(texts)
    return np.array(text_embeddings)

In [ ]:
def embed_cluster_texts(texts: list[str]) -> pandas.DataFrame:
    """
    对文本列表进行嵌入和聚类，并返回一个包含文本、嵌入和聚类标签的数据框
    该函数将嵌入生成和聚类结合成一个步骤
    Args:
        texts: 需要处理的文本列表
    Returns:
        返回包含文本、嵌入和聚类标签的数据框，包含三列：
        - text：原始文本
        - embd：嵌入向量
        - cluster：聚类标签（可能是多个）
    说明：
        这个函数是 RAPTOR 流程的核心步骤之一
    """
    text_embeddings_np = embed(texts)
    cluster_labels = perform_clustering(text_embeddings_np, 10, 0.1)
    df = pd.DataFrame()
    df["text"] = texts
    df["embd"] = list(text_embeddings_np)
    df["cluster"] = cluster_labels
    return df

In [ ]:
def fmt_txt(df: pd.DataFrame) -> str:
    """
    将数据框中的文本格式化成单个字符串
    Args:
        df: 需要处理的数据框，内部包含 text、embd、cluster 三个字段
    Returns:
        返回合并格式化后的字符串
    说明：
        将同一聚类的文本合并成一个字符串，用于生成聚类摘要
    """
    unique_txt = df["text"].tolist()
    return "--- --- \n --- ---".join(unique_txt)

In [ ]:
def embed_cluster_summarize_texts(texts: list[str], level: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    对传入的文本列表进行嵌入、聚类和总结
    该函数首先为文本生成嵌入，基于相似性对它们进行聚类，
    扩展聚类分配以便处理，然后总结每个聚类中的内容
    Args:
        texts: 需要处理的文本列表
        level: 一个整数，表示当前处理的层级（用于追踪递归深度）
    Returns:
        包含两个数据框的元组：
        - 第一个 DataFrame (df_clusters)：包括原始文本、它们的嵌入以及聚类分配
        - 第二个 DataFrame (df_summary)：包含每个聚类的摘要信息、指定的处理级别以及聚类标识符
    说明：
        这是 RAPTOR 每一层处理的核心函数
    """
    df_clusters = embed_cluster_texts(texts)
    expanded_list = []
    for index, row in df_clusters.iterrows():
        for cluster in row["cluster"]:
            expanded_list.append(
                {"text": row["text"], "embd": row["embd"], "cluster": cluster}
            )
    expanded_df = pd.DataFrame(expanded_list)
    all_clusters = expanded_df["cluster"].unique()
    template = """Here is a sub-set of LangChain Expression Language doc. 
    LangChain Expression Language provides a way to compose chain in LangChain.
    Give a detailed summary of the documentation provided.
    Documentation:
    {context}
    """
    prompt = ChatPromptTemplate.from_template(template)
    chain = prompt | model | StrOutputParser()
    summaries = []
    for i in all_clusters:
        df_cluster = expanded_df[expanded_df["cluster"] == i]
        formatted_txt = fmt_txt(df_cluster)
        summaries.append(chain.invoke({"context": formatted_txt}))
    df_summary = pd.DataFrame(
        {
            "summaries": summaries,
            "level": [level] * len(summaries),
            "cluster": list(all_clusters),
        }
    )
    return df_clusters, df_summary

In [ ]:
def recursive_embed_cluster_summarize(
        texts: list[str], level: int = 1, n_levels: int = 3,
) -> dict[int, tuple[pd.DataFrame, pd.DataFrame]]:
    """
    递归地嵌入、聚类和总结文本，直到达到指定的级别或唯一聚类数变为 1，
    将结果存储在每个级别处
    这是 RAPTOR 算法的核心递归函数，自底向上构建文档树。
    Args:
        texts: 要处理的文本列表（初始调用时是叶子节点）
        level: 当前递归级别（从 1 开始）
        n_levels: 递归的最大深度（默认为 3）
    Returns:
        一个字典，其中键是递归级别，值是包含该级别处聚类 DataFrame 和总结 DataFrame 的元组
    工作流程：
        Level 1: 原始文档块 -> 聚类 -> 生成摘要
        Level 2: Level 1 的摘要 -> 聚类 -> 生成更高层次的摘要
        Level 3: Level 2 的摘要 -> 聚类 -> 生成最高层次的摘要
    终止条件：
        1. 达到最大层数 n_levels
        2. 只剩下一个聚类（无法继续细分）
    """
    results = {}
    df_clusters, df_summary = embed_cluster_summarize_texts(texts, level)
    results[level] = (df_clusters, df_summary)
    unique_clusters = df_summary["cluster"].nunique()
    if level < n_levels and unique_clusters > 1:
        new_texts = df_summary["summaries"].tolist()
        next_level_results = recursive_embed_cluster_summarize(
            new_texts, level + 1, n_levels
        )
        results.update(next_level_results)
    return results

In [ ]:
loaders = [
    UnstructuredLoader("./流浪地球.txt"),
    UnstructuredLoader("./电商产品数据.txt"),
    UnstructuredLoader("./项目API文档.md"),
]
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=0,
    separators=["\n\n", "\n", "。|！|？", r"\.\s|\!\s|\?\s", r"；|;\s", r"，|,\s", " ", ""],
    is_separator_regex=True,
)
docs = []
for loader in loaders:
    docs.extend(loader.load_and_split(text_splitter))
leaf_texts = [doc.page_content for doc in docs]
results = recursive_embed_cluster_summarize(leaf_texts, level=1, n_levels=3)
all_texts = leaf_texts.copy()
for level in sorted(results.keys()):
    summaries = results[level][1]["summaries"].tolist()
    all_texts.extend(summaries)
db.add_texts(all_texts)
retriever = db.as_retriever(search_type="mmr")
search_docs = retriever.invoke("流浪地球中的人类花了多长时间才流浪到新的恒星系？")
print(search_docs)
print(len(search_docs))